# <a id='toc1_'></a>[relapse](#toc0_)

**enge Definition eines Rezidivs** 
- Filter: lokaler Beurteilung Residualstatus = R0 (UND M <> 1)
- Rezidiv wenn  
  - Gesamtbeurteilung: Y  oder  
  - Lokaler Tumorstatus: R  oder  
  - Tumorstatus Lymphknoten:R oder  
  - Verlauf Fernmetastasen:R   

**erweiterte Definition (Verworfen)**
- Rezidiv wenn
  - (TNM)r_Symbol:r UND 
  - (Folgeereignis T>0 oder Folgeereignis N>0 oder Folgeereignis M>0)
 

**Table of contents**<a id='toc0_'></a>    
- [relapse](#toc1_)    
  - [⚙️ settings](#toc1_1_)    
  - [📆 data as of](#toc1_2_)    
  - [op](#toc1_3_)    
  - [analysis relapse categorization](#toc1_4_)    
  - [bitmask](#toc1_5_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [14]:
import os
from pathlib import Path
import pandas as pd
import duckdb as ddb
from connection_helper import sql
import numpy as np
from pandas_plots import tbl, pls, hlp
import datetime as dt

hlp.show_package_version(["pygwalker"])
os.environ['THEME']='light'
os.environ['DEBUG']='0'

dir_db=Path("C://temp") if hlp.get_os(hlp.OperatingSystem.WINDOWS) else Path(os.path.expanduser("~/tmp"))

file_db_clin = dir_db/'workflow/2025-10-20_data_clin.duckdb'
# file_db_clin = dir_db/'2025-06-24_data_clin.duckdb'

if not file_db_clin.exists():
    raise(FileNotFoundError(f"File {file_db_clin} not found"))

🐍 3.12.8 | 📦 pygwalker: 0.4.9.15 | 📦 pandas: 2.3.3 | 📦 numpy: 1.26.4 | 📦 duckdb: 1.4.1 | 📦 pandas-plots: 0.20.4 | 📦 connection-helper: 0.13.1


## <a id='toc1_1_'></a>[⚙️ settings](#toc0_)

In [15]:
FILTER_DY="z_dy=2020"
FILTER_ICD10="z_icd10_3d in ('C50')"
FILTER_R0="upper(left(op.Lokale_Beurteilung_Residualstatus,2)) = 'R0'"
FILTER_COMB="has_rez_ges_lo_ly_fm_label != '-'"

## <a id='toc1_2_'></a>[📆 data as of](#toc0_)

In [16]:
con = ddb.connect(file_db_clin, read_only=True)

sql.print_meta(file_db_clin)

sqlite db file:          2025-10-20_data_clin.duckdb
data tag:                v2.3
last kkr data import:    2025-09-30
sql table created:       2025-10-20 18:49:45
doi:                     10.18444/5.03.01.0005.0021.0001
document created:        2025-10-31 14:34:28


In [17]:
# n = con.sql("select count(*) from Folgeereignis_TNM").fetchone()[0]
# n_r = con.sql("select count(*) from Folgeereignis_TNM where r_Symbol is not null").fetchone()[0]
# print(f"{n_r:_} Folgeereignisse von {n:_} (alle klin. Daten) haben r_symbol")

## <a id='toc1_3_'></a>[tum vs op](#toc0_)

In [18]:
FILTER = f"{FILTER_DY} and {FILTER_ICD10}"

db_r0 = con.sql(f"""--sql
    with t1 as (
        select  tum.z_tum_id,
                z_kkr_label,
                upper(left(op.Lokale_Beurteilung_Residualstatus,2)) as r_status,
                z_tum_op_count,
        from Tumor tum
        join Patient pat on tum.z_pat_id = pat.oBDS_RKIPatientId
        left join op on op.z_tum_id = tum.z_tum_id
        where {FILTER}
        -- and not (
        --     z_icd10_3d in ('C44','C70','C71','C72')
        --     or left(z_icd10_3d,1) ='D'
        --     or right(z_icd10_3d,2)::int8 between 76 and 97
        -- )
    ),
    t2 as (
        select  z_tum_id,
                any_value(z_kkr_label) as z_kkr_label,
                any_value(z_tum_op_count) as z_tum_op_count,
                max(ifnull(r_status,'')='R0') as has_r_status,
        from t1
        group by z_tum_id
    ),
    t3 as (
        select  *,
                case
                    when z_tum_op_count = 0 then '1_no_op'
                    when not has_r_status then '2_op_no_r0'
                    when has_r_status then '3_op_r0'
                    else '9_unknown'
                end as categ_op_r0
        from t2
    )
    select * from t3
""")

# tbl.descr_db(db_r0
#     # .filter("z_tum_op_count = 0")
#     , "r0")

n_tum = con.sql("""select count(distinct z_tum_id) from db_r0""").fetchone()[0]
n_op = con.sql("""select sum(z_tum_op_count) from db_r0""").fetchone()[0]
print(f"FILTER: {FILTER} | darin {n_op:_} Operationen, {n_tum:_} Tumore")

FILTER: z_dy=2020 and z_icd10_3d in ('C50') | darin 70_309 Operationen, 77_633 Tumore


In [19]:
pls.plot_stacked_bars(
    db_r0.project("z_kkr_label, categ_op_r0").to_df(),
    relative=True,
    show_total=True,
    orientation="h",
    show_pct_bar=True,
    caption="tum_op_r0",
    height=600,
    width=1500,
    kkr_col="z_kkr_label"
    )


## <a id='toc1_4_'></a>[analysis relapse categorization](#toc0_)
- gezählt sind Tumore
- Kategorien
  - `1_fo_relapse` - Tumore mit Rezidiv nach enger Definition
  - `2_fo_relapse_tnm` - Tumore mit Rezidiv nach erweiterter Definition
  - `3_fo_no_relapse` - Tumore mit Folgeereignis ohne o.a. Rezidiv
  - `4_no_fo` - Tumore ohne Folgeereignis
  - `9_unknown` - UNbekannt

In [20]:
FILTER = f"{FILTER_DY} and {FILTER_ICD10} and {FILTER_R0}"
db_rez = con.sql(f"""--sql
    with t1 as (
        select  tum.z_tum_id
                ,z_pat_id
                ,fol.FolgeereignisId
                ,z_kkr_label
                ,z_m_pc_1
                ,fol.Gesamtbeurteilung_Tumorstatus
                ,fol.Verlauf_Lokaler_Tumorstatus 
                ,fol.Verlauf_Tumorstatus_Lymphknoten
                ,fol.Verlauf_Tumorstatus_Fernmetastasen
                ,r_Symbol
                ,T, N, M
        from Tumor tum
        join Patient pat on tum.z_pat_id = pat.oBDS_RKIPatientId
        left join op on op.z_tum_id = tum.z_tum_id
        left join Folgeereignis fol on fol.z_tum_id = tum.z_tum_id
        left join Folgeereignis_TNM tnm on tnm.z_tum_id = tum.z_tum_id
        where {FILTER}
        --and ifnull(z_m_pc_1,'0') <> '1'
    ),
    t2 as (
        select  *
                ,(Gesamtbeurteilung_Tumorstatus ==  'Y') as has_gesamt
                ,(Verlauf_Lokaler_Tumorstatus == 'R') as has_lokal
                ,(Verlauf_Tumorstatus_Lymphknoten == 'R') as has_lymph
                ,(Verlauf_Tumorstatus_Fernmetastasen == 'R') as has_fm
                -- extended
                ,(r_Symbol is not null) as has_r_symbol
                ,(left(T,1) in ('1','2','3','4')) as has_t2plus
                ,(left(N,1) in ('1','2','3','4')) as has_n2plus
                ,(left(M,1) in ('1','2','3','4')) as has_m2plus
        from t1
    ),
    t3 as (
        select  z_tum_id
                ,first(z_pat_id) as z_pat_id
                ,first(FolgeereignisId) as FolgeereignisId
                ,first(z_m_pc_1) as z_m_pc_1
                ,first(z_kkr_label) as z_kkr_label
                --,max(has_r0)::text as has_r0
                ,max(has_gesamt)::text as has_gesamt
                ,max(has_lokal)::text as has_lokal
                ,max(has_lymph)::text as has_lymph
                ,max(has_fm)::text as has_fm
                ,max(has_r_symbol)::text as has_r_symbol
                ,max(has_t2plus)::text as has_t2plus
                ,max(has_n2plus)::text as has_n2plus
                ,max(has_m2plus)::text as has_m2plus
        from t2
        group by z_tum_id
    ),
    t4 as (
        select  t3.*
                ,pat.Verstorben
                ,case when (has_gesamt or has_lokal or has_lymph or has_fm) then true else false end as has_rel1
                ,case when (has_gesamt or has_lokal or has_lymph or has_fm) or ((has_t2plus or has_n2plus or has_m2plus)) then true else false end as has_rel2
                ,case 
                    when FolgeereignisId is not null and (has_gesamt or has_lokal or has_lymph or has_fm) then '1_fo_relapse'
                    when FolgeereignisId is not null and has_r_symbol and (has_t2plus or has_n2plus or has_m2plus) then '2_fo_relapse_tnm'
                    --when FolgeereignisId is not null and Verstorben = 'J' then '3_fo_deceased'
                    when FolgeereignisId is not null then '3_fo_no_relapse'
                    when FolgeereignisId is null then '4_no_fo'
                    else '9_unknown'
                end as categ_relapse
        from t3
        join Patient pat on pat.oBDS_RkiPatientId = t3.z_pat_id
    )
    select * from t4
""")
# tbl.descr_db(db_rez, "relapse")
n_tum = con.sql("select count(*) from db_rez").fetchone()[0]
print(f"FILTER: {FILTER} | darin {n_tum:_} Tumore")

FILTER: z_dy=2020 and z_icd10_3d in ('C50') and upper(left(op.Lokale_Beurteilung_Residualstatus,2)) = 'R0' | darin 44_976 Tumore


In [21]:
pls.plot_stacked_bars(
    db_rez.project("z_kkr_label, categ_relapse").to_df(),
    relative=True,
    show_total=True,
    orientation="h",
    show_pct_bar=True,
    caption="tum_relapse",
    height=600,
    width=1600,
    kkr_col="z_kkr_label"
)

tbl.pivot_df(db_rez.project("z_kkr_label, categ_relapse").to_df(), swap=False, )

categ_relapse,1_fo_relapse,2_fo_relapse_tnm,3_fo_no_relapse,4_no_fo,Total
z_kkr_label,,,,,
02-HH,64 (0.1%),0,162 (0.4%),962 (2.1%),1_188 (2.6%)
05-NW,464 (1.0%),22 (0.0%),5_186 (11.5%),4_907 (10.9%),10_579 (23.5%)
06-HE,195 (0.4%),0,2_005 (4.5%),1_666 (3.7%),3_866 (8.6%)
07-RP,74 (0.2%),0,1_721 (3.8%),806 (1.8%),2_601 (5.8%)
08-BW,588 (1.3%),0,5_707 (12.7%),1_229 (2.7%),7_524 (16.7%)
09-BY,503 (1.1%),8 (0.0%),3_056 (6.8%),4_175 (9.3%),7_742 (17.2%)
11-BE,163 (0.4%),3 (0.0%),1_231 (2.7%),944 (2.1%),2_341 (5.2%)
12-BB,139 (0.3%),0,1_048 (2.3%),810 (1.8%),1_997 (4.4%)
13-MV,109 (0.2%),1 (0.0%),1_026 (2.3%),107 (0.2%),1_243 (2.8%)


In [22]:
if os.environ.get("DEBUG") == "1":
    db_rez_filter = db_rez.filter("left(categ_relapse,1) = '3' and z_kkr_label = '15-ST'")
    display(db_rez_filter)
    hlp.get_tum_details("961802ce-2afc-46a0-9e77-e70c4aaa9730", con)
    # display(con.sql("select * from Bestrahlung where STId in (select STId from db_st_filter) order by STId"))

## <a id='toc1_5_'></a>[combinations](#toc0_)

In [23]:
FILTER = f"{FILTER_DY} and {FILTER_ICD10} and {FILTER_COMB}"

db_fo = (con.sql(f"""--sql
    with t1 as (
        select  tum.z_tum_id,
                z_kkr_label,
                (Gesamtbeurteilung_Tumorstatus = 'Y')::tinyint as has_rez_gesamt,
                (Verlauf_Lokaler_Tumorstatus = 'R')::tinyint as has_rez_lokal,
                (Verlauf_Tumorstatus_Lymphknoten = 'R')::tinyint as has_rez_lymph,
                (Verlauf_Tumorstatus_Fernmetastasen = 'R')::tinyint as has_rez_fm,
        from Folgeereignis fo
        join Tumor tum on fo.z_tum_id = tum.z_tum_id
        where {FILTER_DY} and {FILTER_ICD10}
    ),
    t2 as (
        select  z_tum_id,
                any_value(z_kkr_label) as z_kkr_label,
                max(has_rez_gesamt) as has_rez_gesamt,
                max(has_rez_lokal) as has_rez_lokal,
                max(has_rez_lymph) as has_rez_lymph,
                max(has_rez_fm) as has_rez_fm,
                max(has_rez_gesamt) + max(has_rez_lokal) * 2 + max(has_rez_lymph) * 4 + max(has_rez_fm) * 8 as has_rez_ges_lo_ly_fm
        from t1
        group by z_tum_id
    ) -- bitmask
    -- t3 as (
    --
    -- )
    select * from t2
    """)
    .add_bitmask_label(con=con, bitmask_col="has_rez_ges_lo_ly_fm", labels=["gesamt", "lokal", "lymph", "fm"])
    .filter(f"{FILTER_COMB}")
)
n_fo = con.sql("""select count(*) from db_fo""").fetchone()[0]
print(f"FILTER: {FILTER} | darin {n_fo:_} Folgeereignisse")
# tbl.descr_db(db_fo, "fo")

FILTER: z_dy=2020 and z_icd10_3d in ('C50') and has_rez_ges_lo_ly_fm_label != '-' | darin 3_247 Folgeereignisse


In [24]:
pls.plot_bars(
    db_fo.project("has_rez_ges_lo_ly_fm_label").to_df(),
    orientation="h",
    caption="tum_fo",
    height=600,
    width=1500,
    )

In [25]:
pls.plot_stacked_bars(
    db_fo.project("z_kkr_label, has_rez_ges_lo_ly_fm_label").to_df(),
    relative=True,
    show_total=True,
    orientation="h",
    show_pct_bar=True,
    caption="tum_fo",
    height=600,
    width=1500,
    kkr_col="z_kkr_label"
    )

In [26]:
if os.environ.get("DEBUG") == "1":
    tbl.describe_df(
        db_rez
        .project(
            "* exclude(z_tum_id, z_pat_id, FolgeereignisId)",
        ).to_df(),
        "relapse",
    )